# Занятие 28. Практика: признаки для цены квартиры (~90 мин)

Вы **пишете весь код сами**. Ячейку **«Дано»** не меняйте.

Задача — регрессия: предсказать **цену** квартиры (млн руб.) по объявлению. Модель для проверки идей — kNN-регрессия (занятие 25), метрика — **MAE**. Теория — `feature_engineering_theory.ipynb` (занятие 27).

Ориентир по времени указан у каждого задания. Если застряли — идите дальше и вернитесь позже.

---
## Дано: датасет

Синтетическая таблица объявлений (300 строк): площадь, комнаты, район, состояние, балкон, дата публикации, просмотры за месяц, доход района (есть пропуски) и **цена** — целевая переменная. Момент прогноза — `PREDICTION_DATE`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

districts = rng.choice(['центр', 'спальный', 'пригород'], size=n, p=[0.3, 0.5, 0.2])
condition = rng.choice(['требует ремонта', 'среднее', 'хорошее'], size=n, p=[0.2, 0.5, 0.3])
area = rng.uniform(25, 130, n).round(0)
rooms = np.clip((area / 30).astype(int) + rng.integers(0, 2, n), 1, 5)
balcony = rng.integers(0, 2, n)
listing_date = pd.Timestamp('2026-01-01') + pd.to_timedelta(rng.integers(0, 180, n), unit='D')
views = np.expm1(rng.normal(4.0, 1.2, n)).round(0)

district_bonus = pd.Series(districts).map({'центр': 4.0, 'спальный': 1.0, 'пригород': 0.0}).values
condition_bonus = pd.Series(condition).map(
    {'требует ремонта': -1.5, 'среднее': 0.0, 'хорошее': 1.5}
).values
price = (0.13 * area + 0.02 * area * balcony + district_bonus
         + condition_bonus + rng.normal(0, 1.0, n)).round(1)

district_income = pd.Series(districts).map({'центр': 95.0, 'спальный': 60.0, 'пригород': 45.0}).values
district_income = district_income + rng.normal(0, 5, n).round(0)
missing_mask = rng.random(n) < 0.15
district_income[missing_mask] = np.nan

flats = pd.DataFrame({
    'площадь': area,
    'комнаты': rooms,
    'район': districts,
    'состояние': condition,
    'балкон': balcony,
    'дата': listing_date,
    'просмотры_за_месяц': views,
    'доход_района': district_income,
    'цена': price,
})

PREDICTION_DATE = pd.Timestamp('2026-07-01')
RANDOM_STATE = 42
print('Объектов:', len(flats))
flats.head()


---
## Задание 0. Постановка (~5 мин)

По мотивам п. 1 теории.

1. Создайте словарь `task_spec` с ключами `объект`, `цель`, `тип_задачи`, `момент_прогноза`.
2. Назовите в комментарии один признак, который **нельзя** строить (утечка), и почему.

**Критерий:** словарь заполнен; пример утечки назван.

---
## Задание 1. Split (~7 мин)

По мотивам п. 0 теории (и занятия 25, п. 4).

1. Разбейте `flats` на `flats_train` и `flats_val` (70/30, `random_state=RANDOM_STATE`).
2. Выведите размеры.

**Критерий:** 210 / 90 объектов; дальше всё «обучаемое» — только по train.

---
## Задание 2. Числовые признаки (~10 мин)

По мотивам пп. 2–3 теории. В **обеих** таблицах (train и val):

1. `площадь_на_комнату` = площадь / комнаты.
2. `log_просмотры` = `np.log1p(просмотры_за_месяц)`.

**Критерий:** новые столбцы есть в train и val; значения без NaN.

---
## Задание 3. Признаки из даты (~10 мин)

По мотивам пп. 7–8 теории. В обеих таблицах:

1. `месяц` из даты; `месяц_sin`, `месяц_cos` (цикличность).
2. `дней_с_публикации` = `PREDICTION_DATE` − дата.

**Критерий:** `дней_с_публикации` ≥ 0 для всех строк.

---
## Задание 4. Категории (~12 мин)

По мотивам пп. 9–11 теории.

1. `OneHotEncoder(handle_unknown='ignore')` для `район`: `fit` на train, `transform` на train и val.
2. `OrdinalEncoder` для `состояние` с явным порядком: требует ремонта < среднее < хорошее; столбец `состояние_код` в обеих таблицах.

**Критерий:** кодировщики обучены только на train; в val нет ошибок.

---
## Задание 5. Пропуски (~8 мин)

По мотивам п. 12 теории.

1. `доход_пропущен` — индикатор 0/1 в обеих таблицах.
2. `доход_заполнен` — NaN заменены **медианой train**.

**Критерий:** медиана посчитана только по train; NaN не осталось.

---
## Задание 6. Baseline-набор и kNN (~10 мин)

По мотивам пп. 15–16 теории.

1. Базовый набор: `['площадь', 'комнаты']`.
2. Pipeline: `StandardScaler` + `KNeighborsRegressor(n_neighbors=5)`.
3. Обучите на train, посчитайте **MAE на validation** → `mae_base`.

**Критерий:** scaler обучается внутри pipeline только на train.

---
## Задание 7. Добавляем группы признаков (~12 мин)

По мотивам п. 16 теории. Сравните на validation наборы (каждый = базовый + одна группа):

- `+ ['площадь_на_комнату', 'log_просмотры']`
- `+ ['состояние_код', 'балкон']`
- `+ one-hot районов`

**Критерий:** таблица «набор → MAE»; выводы о том, какая группа помогла.

---
## Задание 8. Комбинированный набор (~8 мин)

1. Соберите набор из базового + всех групп, которые **улучшили** MAE в задании 7.
2. Посчитайте `mae_final` на validation и сравните с `mae_base`.

**Критерий:** `mae_final` ≤ `mae_base`; состав набора обоснован результатами задания 7.

---
## Задание 9. Итог (~8 мин)

В markdown-ячейке ниже ответьте:

1. Какие группы признаков улучшили MAE, какие нет — и почему (гипотеза)?
2. Какой признак вы **не стали** строить из-за утечки?
3. Что нужно не забыть при применении модели к завтрашним объявлениям (кодировщики, медиана, масштаб)?

**Критерий:** ответы ссылаются на числа из заданий 7–8, а не «кажется».

*Ответ студента.*